<a href="https://colab.research.google.com/github/hdev14/tech-challenger-03/blob/main/FIAP_TECH_CHALLENGER_FASE_3_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

In [ ]:
!pip install -q langchain-community jq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.2/770.2 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.


In [ ]:
from langchain_community.document_loaders import JSONLoader

def metadata_func(record: dict, metadata: dict) -> dict:
    metadata["file_source"] = record.get("file_source", "")
    return metadata

loader = JSONLoader(
    file_path='/content/drive/MyDrive/FIAP/cancer_QA_treated.json',
    jq_schema='.[]',
    content_key='answer', # Setting answer as the main content, but we can also use custom mapping
    metadata_func=metadata_func
)

documents = loader.load()

print(f"Successfully loaded {len(documents)} documents using LangChain JSONLoader.")

Successfully loaded 729 documents using LangChain JSONLoader.


In [ ]:
import chromadb
from chromadb.utils import embedding_functions
import os

persist_directory = '/content/drive/MyDrive/FIAP/chroma_db'
os.makedirs(persist_directory, exist_ok=True)

chroma_client = chromadb.PersistentClient(path=persist_directory)

embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

collection = chroma_client.get_or_create_collection(
    name="cancer_qa_collection",
    embedding_function=embedding_fn
)

db_documents = []
db_metadatas = []
db_ids = []

for index, doc_item in enumerate(documents):
    text_content = doc_item.page_content
    file_source = doc_item.metadata.get('file_source', '')

    db_documents.append(text_content)
    db_metadatas.append({ "file_source": file_source })
    db_ids.append(f"doc_{index}")

batch_size = 500
for i in range(0, len(db_documents), batch_size):
    batch_docs = db_documents[i:i+batch_size]
    batch_meta = db_metadatas[i:i+batch_size]
    batch_ids = db_ids[i:i+batch_size]
    collection.add(
        documents=batch_docs,
        metadatas=batch_meta,
        ids=batch_ids
    )

print(f"Successfully saved and added {collection.count()} documents to the Chroma DB collection in Google Drive using LangChain documents.")

Successfully saved and added 729 documents to the Chroma DB collection in Google Drive using LangChain documents.


In [ ]:
test_query = "What are the main symptoms of cancer?"

results = results = collection.query(
  query_texts=[test_query],
  n_results=3
)

print("Search Results for query:", test_query)
for i in range(len(results['documents'][0])):
    doc = results['documents'][0][i]
    meta = results['metadatas'][0][i]
    print(f"\n--- Result {i+1} ---")
    print(f"Document: {doc}")
    print(f"Source: {meta.get('file_source')}")

Search Results for query: What are the main symptoms of cancer?

--- Result 1 ---
Document: Question: What are the symptoms of Colon Cancer ?
Answer: Signs of colon cancer include blood in the stool or a change in bowel habits. These and other signs and symptoms may be caused by colon cancer or by other conditions. Check with your doctor if you have any of the following:         - A change in bowel habits.    -  Blood (either bright red or very dark) in the stool.    -  Diarrhea, constipation, or feeling that the bowel does not empty all the way.    - Stools that are narrower than usual.    - Frequent gas pains, bloating, fullness, or cramps.    - Weight loss for no known reason.    - Feeling very tired.    -  Vomiting.
Source: 0000037_1.xml

--- Result 2 ---
Document: Question: What are the symptoms of Adult Primary Liver Cancer ?
Answer: Signs and symptoms of adult primary liver cancer include a lump or pain on the right side. These and other signs and symptoms may be caused by adult